**Requerimientos Detallados del Enunciado


Módulo A: Abstracción y Encapsulamiento (Producto)
Implementar la clase base abstracta Producto, que hereda de abc.ABC.


Atributos protegidos: _nombre (str) y _precio_base (float).

Propiedad @property: Crear un getter y un setter para precio_base. El setter debe lanzar un ValueError si el precio asignado es menor o igual a cero.

Método abstracto @abstractmethod def calcular_precio_final(self) -> float: Firma obligatoria para calcular el costo real de venta.

Método abstracto @abstractmethod def obtener_detalle(self) -> str: Firma obligatoria que devuelva una cadena descriptiva para la impresión del ticket.

Método especial __str__: Retornar una representación legible del producto con su nombre y precio base formateado.


Modulo B: Implementar la clase concreta Bebida que hereda de Producto:


Atributos adicionales: tamano ("Pequeño", "Mediano", "Grande") y tipo_leche ("Entera", "Descremada", "Vegetal").

Reglas de Negocio en calcular_precio_final():

Tamaño Pequeño: no añade recargo al precio_base.

Tamaño Mediano: añade un recargo del +15% sobre el precio base.

Tamaño Grande: añade un recargo del +30% sobre el precio base.

Leche Vegetal: añade un costo fijo adicional de $500.

Sobrescribir obtener_detalle() para reflejar el tamaño, tipo de leche y el precio final calculado.

Implementar la clase concreta Alimento que hereda de Producto:

Atributos adicionales: calentar (bool) y es_vegano (bool).

Reglas de Negocio en calcular_precio_final():

Si requiere servicio de calentado en horno (calentar=True), se cobra un recargo por servicio de $200.

Todo alimento tiene un impuesto de elaboración local del +5% sobre el precio base.

Sobrescribir obtener_detalle() especificando si va caliente/frío y si es apto para veganos.

Módulo C: Composición y Delegación (Cliente, ItemPedido y Pedido)

Implementar la clase Cliente:
Atributos: nombre (str), rut (str) y tipo_cliente ("Regular", "Frecuente", "VIP").

Método obtener_porcentaje_descuento() -> float:

"Regular": 0.0 (0%).

"Frecuente": 0.10 (10%).

"VIP": 0.20 (20%).

Implementar la clase ItemPedido (Composición de línea):

Contiene una instancia de un Producto (composición débil/asociación) y una cantidad (int).

Validación: Validar con @property que la cantidad sea un entero mayor o igual a 1.

Método calcular_subtotal() -> float: Multiplica la cantidad por el resultado polimórfico de self.producto.calcular_precio_final().

Implementar la clase Pedido (Composición fuerte del ticket):

Posee un identificador único numérico (id_pedido), una instancia de Cliente y una lista interna protegida de líneas de compra: self._items = [].

Método agregar_item(producto: Producto, cantidad: int): Crea internamente la instancia de ItemPedido y la añade a la lista.

Método calcular_subtotal() -> float: Itera sobre _items sumando los subtotales de cada línea (delegación de responsabilidad).

Método calcular_descuento() -> float: Aplica el porcentaje correspondiente obtenido desde el objeto Cliente sobre el subtotal.

Método calcular_total() -> float: Resta el descuento al subtotal acumulado.

Método emitir_boleta() -> str: Genera un texto con el encabezado del cliente, el desglose línea por línea invocando obtener_detalle(), y los totales finales formateados.

Módulo D: Gestión Global de Turno (CajaRegistradora)

Implementar la clase CajaRegistradora:

Inicializa con un identificador de caja, el nombre del cajero responsable y una lista interna self._pedidos_procesados = [].

Método registrar_pedido(pedido: Pedido): Agrega un pedido completado a la colección histórica del turno.

Método generar_cierre_caja() -> dict: Retorna un informe estadístico consolidado que contenga:

Cantidad de pedidos atendidos.

Recaudación bruta total y monto total de descuentos otorgados.

Ticket promedio de venta.

Pedido con el monto de venta más alto registrado en el turno **

In [1]:
"""
Sistema de Punto de Venta para Cafetería
Módulos:
    A: Abstracción y Encapsulamiento (Producto)
    B: Especialización y Herencia (Bebida y Alimento)
    C: Composición y Delegación (Cliente, ItemPedido y Pedido)
    D: Gestión Global de Turno (CajaRegistradora)
"""

from abc import ABC, abstractmethod


def formatear_monto(valor: float) -> str:
    """Formatea un monto como pesos chilenos: $1.234"""
    return "$" + f"{valor:,.0f}".replace(",", ".")



# MÓDULO A: Abstracción y Encapsulamiento (Producto)
class Producto(ABC):
    """Clase base abstracta para todos los productos de la cafetería."""

    def __init__(self, nombre: str, precio_base: float):
        self._nombre = nombre
        self._precio_base = None
        self.precio_base = precio_base  # pasa por el setter (validación)

    @property
    def precio_base(self) -> float:
        return self._precio_base

    @precio_base.setter
    def precio_base(self, valor: float):
        if valor <= 0:
            raise ValueError("El precio base debe ser mayor a cero.")
        self._precio_base = float(valor)

    @abstractmethod
    def calcular_precio_final(self) -> float:
        """Calcula el costo real de venta del producto."""

    @abstractmethod
    def obtener_detalle(self) -> str:
        """Devuelve una cadena descriptiva para la impresión del ticket."""

    def __str__(self) -> str:
        return f"{self._nombre} - Precio base: {formatear_monto(self._precio_base)}"


# MÓDULO B: Especialización y Herencia (Bebida y Alimento)
class Bebida(Producto):
    TAMANOS = {"Pequeño": 0.0, "Mediano": 0.15, "Grande": 0.30}
    TIPOS_LECHE = ("Entera", "Descremada", "Vegetal")
    COSTO_LECHE_VEGETAL = 500

    def __init__(self, nombre: str, precio_base: float, tamano: str, tipo_leche: str):
        super().__init__(nombre, precio_base)
        if tamano not in self.TAMANOS:
            raise ValueError(f"Tamaño inválido: {tamano}. Opciones: {list(self.TAMANOS)}")
        if tipo_leche not in self.TIPOS_LECHE:
            raise ValueError(f"Tipo de leche inválido: {tipo_leche}. Opciones: {list(self.TIPOS_LECHE)}")
        self.tamano = tamano
        self.tipo_leche = tipo_leche

    def calcular_precio_final(self) -> float:
        precio = self._precio_base * (1 + self.TAMANOS[self.tamano])
        if self.tipo_leche == "Vegetal":
            precio += self.COSTO_LECHE_VEGETAL
        return precio

    def obtener_detalle(self) -> str:
        return (f"{self._nombre} (Tamaño: {self.tamano}, Leche: {self.tipo_leche}) "
                f"- {formatear_monto(self.calcular_precio_final())}")


class Alimento(Producto):
    RECARGO_CALENTADO = 200
    IMPUESTO_ELABORACION = 0.05

    def __init__(self, nombre: str, precio_base: float, calentar: bool, es_vegano: bool):
        super().__init__(nombre, precio_base)
        self.calentar = calentar
        self.es_vegano = es_vegano

    def calcular_precio_final(self) -> float:
        precio = self._precio_base * (1 + self.IMPUESTO_ELABORACION)
        if self.calentar:
            precio += self.RECARGO_CALENTADO
        return precio

    def obtener_detalle(self) -> str:
        temperatura = "Caliente" if self.calentar else "Frío"
        vegano = "Apto para veganos" if self.es_vegano else "No apto para veganos"
        return (f"{self._nombre} ({temperatura}, {vegano}) "
                f"- {formatear_monto(self.calcular_precio_final())}")


# MÓDULO C: Composición y Delegación (Cliente, ItemPedido y Pedido)
class Cliente:
    DESCUENTOS = {"Regular": 0.0, "Frecuente": 0.10, "VIP": 0.20}

    def __init__(self, nombre: str, rut: str, tipo_cliente: str = "Regular"):
        if tipo_cliente not in self.DESCUENTOS:
            raise ValueError(f"Tipo de cliente inválido: {tipo_cliente}. "
                             f"Opciones: {list(self.DESCUENTOS)}")
        self.nombre = nombre
        self.rut = rut
        self.tipo_cliente = tipo_cliente

    def obtener_porcentaje_descuento(self) -> float:
        return self.DESCUENTOS[self.tipo_cliente]


class ItemPedido:
    """Línea de compra: asociación con un Producto y una cantidad."""

    def __init__(self, producto: Producto, cantidad: int):
        self.producto = producto
        self._cantidad = None
        self.cantidad = cantidad  # pasa por el setter (validación)

    @property
    def cantidad(self) -> int:
        return self._cantidad

    @cantidad.setter
    def cantidad(self, valor: int):
        if not isinstance(valor, int) or isinstance(valor, bool) or valor < 1:
            raise ValueError("La cantidad debe ser un entero mayor o igual a 1.")
        self._cantidad = valor

    def calcular_subtotal(self) -> float:
        # Polimorfismo: cada producto calcula su propio precio final
        return self._cantidad * self.producto.calcular_precio_final()


class Pedido:
    """Ticket completo: composición fuerte de sus líneas (ItemPedido)."""

    def __init__(self, id_pedido: int, cliente: Cliente):
        self.id_pedido = id_pedido
        self.cliente = cliente
        self._items = []

    def agregar_item(self, producto: Producto, cantidad: int):
        # Los ItemPedido se crean y viven dentro del Pedido
        self._items.append(ItemPedido(producto, cantidad))

    def calcular_subtotal(self) -> float:
        return sum(item.calcular_subtotal() for item in self._items)

    def calcular_descuento(self) -> float:
        return self.calcular_subtotal() * self.cliente.obtener_porcentaje_descuento()

    def calcular_total(self) -> float:
        return self.calcular_subtotal() - self.calcular_descuento()

    def emitir_boleta(self) -> str:
        porcentaje = int(self.cliente.obtener_porcentaje_descuento() * 100)
        lineas = [
            "=" * 55,
            f"BOLETA N° {self.id_pedido}",
            f"Cliente: {self.cliente.nombre} | RUT: {self.cliente.rut}",
            f"Tipo de cliente: {self.cliente.tipo_cliente}",
            "-" * 55,
        ]
        for item in self._items:
            lineas.append(f"{item.cantidad} x {item.producto.obtener_detalle()}")
            lineas.append(f"      Subtotal línea: {formatear_monto(item.calcular_subtotal())}")
        lineas += [
            "-" * 55,
            f"Subtotal:              {formatear_monto(self.calcular_subtotal())}",
            f"Descuento ({porcentaje}%):      -{formatear_monto(self.calcular_descuento())}",
            f"TOTAL A PAGAR:         {formatear_monto(self.calcular_total())}",
            "=" * 55,
        ]
        return "\n".join(lineas)


# MÓDULO D: Gestión Global de Turno (CajaRegistradora)
class CajaRegistradora:
    def __init__(self, id_caja: str, cajero: str):
        self.id_caja = id_caja
        self.cajero = cajero
        self._pedidos_procesados = []

    def registrar_pedido(self, pedido: Pedido):
        self._pedidos_procesados.append(pedido)

    def generar_cierre_caja(self) -> dict:
        cantidad = len(self._pedidos_procesados)
        if cantidad == 0:
            return {
                "cantidad_pedidos": 0,
                "recaudacion_bruta": 0.0,
                "total_descuentos": 0.0,
                "ticket_promedio": 0.0,
                "pedido_mayor_venta": None,
            }

        recaudacion_bruta = sum(p.calcular_subtotal() for p in self._pedidos_procesados)
        total_descuentos = sum(p.calcular_descuento() for p in self._pedidos_procesados)
        total_neto = sum(p.calcular_total() for p in self._pedidos_procesados)
        mayor = max(self._pedidos_procesados, key=lambda p: p.calcular_total())

        return {
            "cantidad_pedidos": cantidad,
            "recaudacion_bruta": recaudacion_bruta,
            "total_descuentos": total_descuentos,
            "ticket_promedio": total_neto / cantidad,
            "pedido_mayor_venta": mayor,
        }


# DEMOSTRACIÓN
if __name__ == "__main__":
    # Productos
    latte = Bebida("Latte", 2500, "Grande", "Vegetal")
    capuchino = Bebida("Capuchino", 2200, "Mediano", "Entera")
    espresso = Bebida("Espresso", 1800, "Pequeño", "Descremada")
    sandwich = Alimento("Sándwich de pavo", 3500, calentar=True, es_vegano=False)
    croissant = Alimento("Croissant de palta", 2800, calentar=False, es_vegano=True)

    print(latte)
    try:
        latte.precio_base = 0
    except ValueError as e:
        print(f"Error esperado: {e}\n")

    # Clientes
    ana = Cliente("Ana Pérez", "12.345.678-9", "VIP")
    luis = Cliente("Luis Soto", "9.876.543-2", "Frecuente")
    marta = Cliente("Marta Rojas", "15.111.222-3", "Regular")

    # Pedidos
    p1 = Pedido(1, ana)
    p1.agregar_item(latte, 2)
    p1.agregar_item(sandwich, 1)

    p2 = Pedido(2, luis)
    p2.agregar_item(capuchino, 1)
    p2.agregar_item(croissant, 2)

    p3 = Pedido(3, marta)
    p3.agregar_item(espresso, 3)

    # Caja
    caja = CajaRegistradora("CAJA-01", "Carlos Muñoz")
    for p in (p1, p2, p3):
        print(p.emitir_boleta())
        print()
        caja.registrar_pedido(p)

    cierre = caja.generar_cierre_caja()
    print(f"CIERRE DE CAJA {caja.id_caja} - Cajero: {caja.cajero}")
    print(f"Pedidos atendidos:    {cierre['cantidad_pedidos']}")
    print(f"Recaudación bruta:    {formatear_monto(cierre['recaudacion_bruta'])}")
    print(f"Descuentos otorgados: {formatear_monto(cierre['total_descuentos'])}")
    print(f"Ticket promedio:      {formatear_monto(cierre['ticket_promedio'])}")
    mayor = cierre["pedido_mayor_venta"]
    print(f"Mayor venta:          Pedido N° {mayor.id_pedido} "
          f"({formatear_monto(mayor.calcular_total())})")

Latte - Precio base: $2.500
Error esperado: El precio base debe ser mayor a cero.

BOLETA N° 1
Cliente: Ana Pérez | RUT: 12.345.678-9
Tipo de cliente: VIP
-------------------------------------------------------
2 x Latte (Tamaño: Grande, Leche: Vegetal) - $3.750
      Subtotal línea: $7.500
1 x Sándwich de pavo (Caliente, No apto para veganos) - $3.875
      Subtotal línea: $3.875
-------------------------------------------------------
Subtotal:              $11.375
Descuento (20%):      -$2.275
TOTAL A PAGAR:         $9.100

BOLETA N° 2
Cliente: Luis Soto | RUT: 9.876.543-2
Tipo de cliente: Frecuente
-------------------------------------------------------
1 x Capuchino (Tamaño: Mediano, Leche: Entera) - $2.530
      Subtotal línea: $2.530
2 x Croissant de palta (Frío, Apto para veganos) - $2.940
      Subtotal línea: $5.880
-------------------------------------------------------
Subtotal:              $8.410
Descuento (10%):      -$841
TOTAL A PAGAR:         $7.569

BOLETA N° 3
Client